In [15]:
import pandas as pd
import sqlite3

In [16]:
# Load 
df = pd.read_csv("market_data_analytics_dataset.csv")

print("Initial Data:")
print(df.head())

Initial Data:
  Vendor_ID Asset    Price            Timestamp Market Currency  Volume  \
0  Vendor_A   TCS  3475.76  2025-01-01 09:15:00    BSE      INR   84810   
1  Vendor_B   TCS  3470.68  2025-01-01 09:15:00    BSE      INR   84463   
2  Vendor_C   TCS  3477.12  2025-01-01 09:15:00    BSE      INR   54124   
3  Vendor_D   TCS  3486.86  2025-01-01 09:15:00    BSE      INR    2529   
4  Vendor_A  INFY  1515.11  2025-01-01 09:15:00    BSE      INR   14535   

   Latency_ms  
0         233  
1         459  
2         121  
3         454  
4          35  


In [17]:
# Remove duplicates
df = df.drop_duplicates()

# Remove null values
df = df.dropna()

# Convert timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

print("\nCleaned Data:")
print(df.info())


Cleaned Data:
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Vendor_ID   3000 non-null   str           
 1   Asset       3000 non-null   str           
 2   Price       3000 non-null   float64       
 3   Timestamp   3000 non-null   datetime64[us]
 4   Market      3000 non-null   str           
 5   Currency    3000 non-null   str           
 6   Volume      3000 non-null   int64         
 7   Latency_ms  3000 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(2), str(4)
memory usage: 187.6 KB
None


In [18]:
#Average price per Asset per Timestamp
df['avg_price'] = df.groupby(['Asset', 'Timestamp'])['Price'].transform('mean')
#Price deviation
df['deviation'] = df['Price'] - df['avg_price']

In [19]:
#Accuracy score
df['accuracy'] = (1 - abs(df['deviation']) / df['avg_price']) * 100

In [20]:
#outlier detection
threshold = 5  
df['is_outlier'] = df['deviation'].abs() > threshold

In [23]:
conn=sqlite3.connect("market_data_analytics_dataset.db")
df.to_sql("processed_prices", conn, if_exists="replace", index=False)

3000

In [24]:
import pandas as pd

query = "SELECT * FROM processed_prices LIMIT 5"
pd.read_sql(query, conn)

,Vendor_ID,Asset,Price,Timestamp,Market,Currency,Volume,Latency_ms,avg_price,deviation,accuracy,is_outlier
0,Vendor_A,TCS,3475.76,2025-01-01 09:15:00,BSE,INR,84810,233,3477.605,-1.845,99.946946,0
1,Vendor_B,TCS,3470.68,2025-01-01 09:15:00,BSE,INR,84463,459,3477.605,-6.925,99.800869,1
2,Vendor_C,TCS,3477.12,2025-01-01 09:15:00,BSE,INR,54124,121,3477.605,-0.485,99.986054,0
3,Vendor_D,TCS,3486.86,2025-01-01 09:15:00,BSE,INR,2529,454,3477.605,9.255,99.733869,1
4,Vendor_A,INFY,1515.11,2025-01-01 09:15:00,BSE,INR,14535,35,1515.185,-0.075,99.995050,0


In [27]:
#best vendor
query = """
SELECT vendor_id, AVG(accuracy) AS avg_accuracy
FROM processed_prices
GROUP BY vendor_id
ORDER BY avg_accuracy DESC;
"""

result = pd.read_sql(query, conn)
print(result)

  Vendor_ID  avg_accuracy
0  Vendor_B     98.898638
1  Vendor_A     98.897181
2  Vendor_D     98.782229
3  Vendor_C     98.761652


In [28]:
#worst vendor
query = """
SELECT vendor_id, AVG(accuracy) AS avg_accuracy
FROM processed_prices
GROUP BY vendor_id
ORDER BY avg_accuracy ASC;
"""

result = pd.read_sql(query, conn)
print(result)

  Vendor_ID  avg_accuracy
0  Vendor_C     98.761652
1  Vendor_D     98.782229
2  Vendor_A     98.897181
3  Vendor_B     98.898638


In [29]:
#most inconsistent vendor
query = """
SELECT vendor_id, 
       AVG(price * price) - AVG(price) * AVG(price) AS variance
FROM processed_prices
GROUP BY vendor_id
ORDER BY variance DESC;
"""

result = pd.read_sql(query, conn)
print(result)

  Vendor_ID      variance
0  Vendor_A  7.231896e+06
1  Vendor_C  7.213345e+06
2  Vendor_D  7.186984e+06
3  Vendor_B  7.167713e+06


In [30]:
#Asset with highest price variation
query = """
SELECT asset, MAX(price) - MIN(price) AS variation
FROM processed_prices
GROUP BY asset
ORDER BY variation DESC;
"""

result = pd.read_sql(query, conn)
print(result)


      Asset  variation
0    MARUTI    2975.54
1       TCS     982.21
2        LT     869.75
3  RELIANCE     707.93
4      HDFC     470.52
5      INFY     444.66
6  AXISBANK     295.96
7     ICICI     254.71
8      SBIN     179.82
9     WIPRO     142.89


In [31]:
#Outlier count per vendor
query = """
SELECT vendor_id, COUNT(*) AS outliers
FROM processed_prices
WHERE is_outlier = 1
GROUP BY vendor_id
ORDER BY outliers DESC;
"""

result = pd.read_sql(query, conn)
print(result)


  Vendor_ID  outliers
0  Vendor_B       303
1  Vendor_D       302
2  Vendor_C       292
3  Vendor_A       286


In [32]:
df.to_csv("processed_prices.csv", index=False)